# Sampled Structure Representative Sampling

This notebook extracts tracked UMA-generated NaCl XSF structures, builds global moment representations, standardizes the feature matrix explicitly, and compares k-means representative sampling with a random baseline. PCA is used to choose a variance-preserving input dimension before t-SNE visualization.

Required optional dependencies: `aenet[torch]`, PyG extensions used by torch featurization, and `aenet[sampling]`.

In [ ]:
from pathlib import Path
import tarfile
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

from aenet.geometry.sampling import random_subset, representative_subset
from aenet.torch_featurize import ChebyshevDescriptor
from aenet.torch_training.dataset import HDF5StructureDataset
from aenet.trainset import TrnSet

RANDOM_SEED = 42
FEATURE_FILE = None  # Set to an HPC .npz output to skip featurization.
NUM_CANDIDATES = 100  # Use None for all 20,000 tracked structures.
NUM_SELECTED = 20  # Use 2,000 for a 10% subset of the full dataset. Can be increased to 20% or 30% if needed.
PCA_VARIANCE_TARGET = 0.95
PCA_N_COMPONENTS_OVERRIDE = None  # Set to an integer after reviewing PCA plots.

if Path("data/NaCl-sampled-structures").exists():
    notebook_dir = Path(".")
else:
    notebook_dir = Path("notebooks")

data_dir = notebook_dir / "data" / "NaCl-sampled-structures"
archive_path = data_dir / "sampled_structures.tar.xz"
if FEATURE_FILE is None and not archive_path.exists():
    raise FileNotFoundError(f"Tracked XSF archive not found: {archive_path}")

The descriptor uses the rounded radial and angular cutoffs supported by the tracked neighbor-shell analysis in `data/NaCl-sampled-structures/cutoff_analysis/`. Set `FEATURE_FILE` to the `.npz` produced by an HPC featurization run to bypass this local build without changing the downstream analysis.

In [ ]:
descriptor = ChebyshevDescriptor(
    species=["Na", "Cl"],
    rad_order=10,
    rad_cutoff=4.8,
    ang_order=3,
    ang_cutoff=3.75,
    min_cutoff=0.5,
    device="cpu",
    dtype=torch.float64,
)

global_moment_settings = {
    "outer_moment": 1,
    "inner_moment": 1,
    "weighted": True,
    "append_weighted": True,
}

if FEATURE_FILE is None:
    archive_workspace = tempfile.TemporaryDirectory(
        prefix="aenet-example-09-"
    )
    workspace_path = Path(archive_workspace.name)
    with tarfile.open(archive_path, mode="r:xz") as archive:
        xsf_members = [
            member
            for member in archive.getmembers()
            if member.isfile() and member.name.endswith(".xsf")
        ]
        if NUM_CANDIDATES is None:
            selected_members = xsf_members
        else:
            selected_names = {
                member.name
                for member in sorted(xsf_members, key=lambda item: item.name)[
                    :NUM_CANDIDATES
                ]
            }
            selected_members = [
                member for member in xsf_members if member.name in selected_names
            ]
        for member in selected_members:
            member_path = Path(member.name)
            if member_path.is_absolute() or ".." in member_path.parts:
                raise ValueError(f"Unsafe archive member: {member.name}")
        archive.extractall(workspace_path, members=selected_members)

    structure_paths = sorted(
        workspace_path.glob("sampled_structures/*.xsf")
    )
    db_path = workspace_path / "representative_sampling.h5"
    dataset = HDF5StructureDataset(
        descriptor=descriptor,
        database_file=str(db_path),
        sources=[str(path) for path in structure_paths],
        mode="build",
    )
    dataset.build_database(show_progress=False, persist_features=True)

    with TrnSet.from_file(db_path) as trnset:
        fingerprints = [
            trnset.read_structure(index).global_moment_fingerprint(
                **global_moment_settings
            )
            for index in range(trnset.num_structures)
        ]

    features = np.vstack(fingerprints)
else:
    feature_file = Path(FEATURE_FILE).expanduser()
    with np.load(feature_file, allow_pickle=False) as data:
        features = np.asarray(data["features"], dtype=float)
        structure_paths = [Path(path) for path in data["paths"].astype(str)]
    if NUM_CANDIDATES is not None:
        features = features[:NUM_CANDIDATES]
        structure_paths = structure_paths[:NUM_CANDIDATES]

if features.shape[0] != len(structure_paths):
    raise ValueError("Feature rows and structure paths must have equal length")
if not 0 < NUM_SELECTED <= len(structure_paths):
    raise ValueError("NUM_SELECTED must be within the candidate count")

print(f"Feature matrix: {features.shape}")
print(f"Selecting {NUM_SELECTED} of {len(structure_paths)} structures")
print("First paths:", [path.name for path in structure_paths[:5]])

The full feature matrix is standardized once, then PCA is fit on all standardized structures before t-SNE. Representative sampling still uses the full standardized feature space, so PCA controls the visualization path rather than discarding information before k-means.


In [ ]:
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

print("Scaled feature matrix:", scaled_features.shape)

K-means representative selection is performed on the full standardized feature matrix. This keeps sampling faithful to the original descriptor representation; PCA-reduced features are reserved for t-SNE visualization.


In [ ]:
representative_indices = representative_subset(
    scaled_features,
    num_structures=NUM_SELECTED,
    random_state=RANDOM_SEED,
)
random_indices = random_subset(
    scaled_features,
    num_structures=NUM_SELECTED,
    random_state=RANDOM_SEED,
)

print("Representative indices:", representative_indices.tolist())
print("Random baseline indices:", random_indices.tolist())

Use the PCA diagnostics below to choose how many standardized feature dimensions to keep before t-SNE. The default keeps the smallest number of principal components that explain at least 90% of the original feature variance; set `PCA_N_COMPONENTS_OVERRIDE` above to choose a different value after inspecting the variance plot.


In [ ]:
pca_full = PCA(random_state=RANDOM_SEED).fit(scaled_features)
explained_variance = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)
pca_components_for_target = int(
    np.searchsorted(cumulative_variance, PCA_VARIANCE_TARGET) + 1
)

if PCA_N_COMPONENTS_OVERRIDE is None:
    n_pca_components = max(2, pca_components_for_target)
else:
    n_pca_components = int(PCA_N_COMPONENTS_OVERRIDE)

if n_pca_components < 2:
    raise ValueError("PCA reduction needs at least 2 components for plotting")
if n_pca_components > scaled_features.shape[1]:
    raise ValueError(
        f"Requested {n_pca_components} PCA components, but only "
        f"{scaled_features.shape[1]} features are available"
    )

pca_scores = pca_full.transform(scaled_features)
pca_reduced_features = pca_scores[:, :n_pca_components]
retained_variance = cumulative_variance[n_pca_components - 1]

print(
    f"{pca_components_for_target} PCA components capture "
    f"{PCA_VARIANCE_TARGET:.0%} of variance"
)
print(
    f"Using {n_pca_components} PCA components for t-SNE "
    f"({retained_variance:.2%} variance retained)"
)

components_to_plot = min(30, len(explained_variance))
component_numbers = np.arange(1, components_to_plot + 1)

fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.bar(
    component_numbers,
    explained_variance[:components_to_plot],
    color="0.72",
    label="Individual variance",
)
ax1.set_xlabel("PCA component")
ax1.set_ylabel("Explained variance ratio")

ax2 = ax1.twinx()
ax2.plot(
    component_numbers,
    cumulative_variance[:components_to_plot],
    marker="o",
    color="tab:blue",
    label="Cumulative variance",
)
ax2.axhline(
    PCA_VARIANCE_TARGET,
    color="tab:red",
    linestyle="--",
    linewidth=1,
    label=f"{PCA_VARIANCE_TARGET:.0%} target",
)
if pca_components_for_target <= components_to_plot:
    ax2.axvline(
        pca_components_for_target,
        color="tab:red",
        linestyle=":",
        linewidth=1,
    )
ax2.set_ylabel("Cumulative explained variance")
ax2.set_ylim(0, 1.02)

handles_1, labels_1 = ax1.get_legend_handles_labels()
handles_2, labels_2 = ax2.get_legend_handles_labels()
ax2.legend(handles_1 + handles_2, labels_1 + labels_2, loc="lower right")
ax1.set_title("PCA variance retained by component count")
fig.tight_layout()


In [ ]:
representative_paths = [structure_paths[index] for index in representative_indices]
random_paths = [structure_paths[index] for index in random_indices]

print("Representative structures:")
for path in representative_paths:
    print(" ", path.name)

print("\nRandom baseline structures:")
for path in random_paths:
    print(" ", path.name)

Temperature labels are taken from the filename suffix, so `snapshot_1234_850K.xsf` is labeled `850K`. These labels show what the full background cloud is made of before highlighting representative structures.


In [ ]:
temperature_labels = np.array([
    path.stem.rsplit("_", 1)[-1]
    for path in structure_paths
])
temperature_order = [
    label
    for label in ["550K", "700K", "850K", "1000K"]
    if label in set(temperature_labels)
]
temperature_colors = {
    "550K": "tab:purple",
    "700K": "tab:green",
    "850K": "tab:orange",
    "1000K": "tab:red",
}

temperature_counts = {
    label: int(np.sum(temperature_labels == label))
    for label in temperature_order
}
print("Temperature counts:", temperature_counts)

The first two PCA components show the strongest linear variance directions in the standardized features. The full background cloud is colored by temperature label, with representative and random selections overlaid.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for label in temperature_order:
    mask = temperature_labels == label
    ax.scatter(
        pca_scores[mask, 0],
        pca_scores[mask, 1],
        s=12,
        alpha=0.55,
        color=temperature_colors[label],
        label=f"{label} ({int(mask.sum())})",
    )
ax.set_xlabel(f"PC1 ({explained_variance[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({explained_variance[1]:.1%} variance)")
ax.legend(title="Temperature", fontsize=8)
fig.tight_layout()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for label in temperature_order:
    mask = temperature_labels == label
    ax.scatter(
        pca_scores[mask, 0],
        pca_scores[mask, 1],
        s=12,
        alpha=0.55,
        color=temperature_colors[label],
        label=f"{label} ({int(mask.sum())})",
    )
ax.scatter(
    pca_scores[representative_indices, 0],
    pca_scores[representative_indices, 1],
    s=70,
    facecolors="none",
    edgecolors="tab:blue",
    linewidths=1.8,
    label="Representative subset",
)
ax.set_xlabel(f"PC1 ({explained_variance[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({explained_variance[1]:.1%} variance)")
ax.legend(title="Temperature", fontsize=8)
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for label in temperature_order:
    mask = temperature_labels == label
    ax.scatter(
        pca_scores[mask, 0],
        pca_scores[mask, 1],
        s=12,
        alpha=0.55,
        color=temperature_colors[label],
        label=f"{label} ({int(mask.sum())})",
    )
ax.scatter(
    pca_scores[random_indices, 0],
    pca_scores[random_indices, 1],
    marker="x",
    s=55,
    color="black",
    linewidths=1.3,
    label="Random baseline",
)
ax.set_xlabel(f"PC1 ({explained_variance[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({explained_variance[1]:.1%} variance)")
ax.legend(title="Temperature", fontsize=8)
fig.tight_layout()

The t-SNE projection below is fit to the PCA-reduced feature matrix. The full background cloud is colored by source temperature, then representative structures are overlaid in blue so you can see which temperature regions they cover.


In [ ]:
n_candidates = pca_reduced_features.shape[0]
if n_candidates < 3:
    raise ValueError("t-SNE visualization needs at least 3 structures")

tsne_perplexity = min(60, max(2, (n_candidates - 1) // 3))
tsne_perplexity = min(tsne_perplexity, n_candidates - 1)

tsne_projection = TSNE(
    n_components=2,
    perplexity=tsne_perplexity,
    init="pca",
    learning_rate="auto",
    random_state=RANDOM_SEED,
).fit_transform(pca_reduced_features)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for label in temperature_order:
    mask = temperature_labels == label
    ax.scatter(
        tsne_projection[mask, 0],
        tsne_projection[mask, 1],
        s=12,
        alpha=0.55,
        color=temperature_colors[label],
        label=f"{label} ({int(mask.sum())})",
    )
ax.set_xlabel("t-SNE component 1")
ax.set_ylabel("t-SNE component 2")
ax.legend(title="Temperature", fontsize=8)
ax.set_title(
    f"t-SNE after {n_pca_components} PCA components "
    f"(perplexity={tsne_perplexity})"
)
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for label in temperature_order:
    mask = temperature_labels == label
    ax.scatter(
        tsne_projection[mask, 0],
        tsne_projection[mask, 1],
        s=12,
        alpha=0.55,
        color=temperature_colors[label],
        label=f"{label} ({int(mask.sum())})",
    )
ax.scatter(
    tsne_projection[representative_indices, 0],
    tsne_projection[representative_indices, 1],
    s=70,
    facecolors="none",
    edgecolors="tab:blue",
    linewidths=1.8,
    label="Representative subset",
)
ax.set_xlabel("t-SNE component 1")
ax.set_ylabel("t-SNE component 2")
ax.legend(title="Temperature", fontsize=8)
ax.set_title(
    f"t-SNE after {n_pca_components} PCA components "
    f"(perplexity={tsne_perplexity})"
)
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for label in temperature_order:
    mask = temperature_labels == label
    ax.scatter(
        tsne_projection[mask, 0],
        tsne_projection[mask, 1],
        s=12,
        alpha=0.55,
        color=temperature_colors[label],
        label=f"{label} ({int(mask.sum())})",
    )
ax.scatter(
    tsne_projection[random_indices, 0],
    tsne_projection[random_indices, 1],
    marker="x",
    s=55,
    color="black",
    linewidths=1.3,
    label="Random baseline",
)
ax.set_xlabel("t-SNE component 1")
ax.set_ylabel("t-SNE component 2")
ax.legend(title="Temperature", fontsize=8)
ax.set_title(
    f"t-SNE after {n_pca_components} PCA components "
    f"(perplexity={tsne_perplexity})"
)
fig.tight_layout()